# 08 — Federated Loss Curves (from saved per-round CSVs)

No training. Scans saved run outputs under ``08-Federated-mlp-lr-rf`` for
``*_train_loss_per_round.csv`` and ``*_per_round.csv`` (``val_loss`` column when
present) and plots train vs. validation loss per run / strategy.

**Coverage note:** older runs saved validation metrics only; train-loss CSVs
exist for the Ceftriaxone ``260922`` run. Rerunning a drug's ``federated.ipynb``
populates the other drugs automatically (the main notebook saves these CSVs).

Output: ``loss_curves_summary.pdf`` + ``loss_curves_summary.csv``.
Runs anywhere (no GPU); runs on Colab like the rest.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed/Processing/Analysis")
else:
    REPO_ROOT = Path("/media/asd/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed/Processing/Analysis")

OUT_DIR = Path("./results_loss_curves_08")
OUT_DIR.mkdir(exist_ok=True)
BASE = REPO_ROOT / "08-Federated-mlp-lr-rf"

run_dirs = sorted(BASE.glob("Drugs/*/[0-9]*-Run/results"))
run_dirs += sorted(BASE.glob("Agg-Analysis/Results/*/results"))
print(f"Found {len(run_dirs)} run result dirs")

In [ ]:
# ── Collect per-round train/val loss records ──
records = []

for d in run_dirs:
    if not d.is_dir():
        continue
    parts = d.parts
    drug = parts[parts.index("Drugs") + 1] if "Drugs" in parts else d.parent.name

    for tp in sorted(d.glob("*_train_loss_per_round.csv")):
        method = tp.name[: -len("_train_loss_per_round.csv")]
        vp = d / f"{method}_per_round.csv"
        records.append({
            "dir": str(d), "drug": drug, "run": d.parent.name, "method": method,
            "train": pd.read_csv(tp),
            "val": pd.read_csv(vp) if vp.exists() else None,
        })

    # val-only fallback (older runs): *per_round.csv with a val_loss column
    covered = {(str(d), r["method"]) for r in records}
    for vp in sorted(d.glob("*_per_round.csv")):
        method = vp.name[: -len("_per_round.csv")]
        if (str(d), method) in covered:
            continue
        df = pd.read_csv(vp)
        if "val_loss" in df.columns:
            records.append({"dir": str(d), "drug": drug, "run": d.parent.name,
                            "method": method + " (val only)", "train": None, "val": df})

if not records:
    print("No loss data found. Rerun a drug's federated.ipynb to generate it.")
else:
    inv = pd.DataFrame([{k: r[k] for k in ("drug", "run", "method")} for r in records])
    print(inv.to_string(index=False))

In [ ]:
# ── Plot + export ──
assert records, "No loss data available."

grouped = {}
for r in records:
    grouped.setdefault(r["dir"], []).append(r)

rows = []
for d, recs in grouped.items():
    fig, ax = plt.subplots(figsize=(8, 5))
    any_curve = False
    for r in recs:
        color = None
        if r["train"] is not None and "All_train_loss" in r["train"].columns:
            ax.plot(r["train"]["round"], r["train"]["All_train_loss"], lw=1.8,
                    label=f"{r['method']} (train)")
            for _, row in r["train"].iterrows():
                rows.append({"drug": r["drug"], "run": r["run"], "method": r["method"],
                             "round": row["round"], "train_loss": row["All_train_loss"],
                             "val_loss": np.nan})
            any_curve = True
        if r["val"] is not None and "val_loss" in r["val"].columns:
            ax.plot(r["val"]["round"], r["val"]["val_loss"], lw=1.5, ls="--", alpha=0.7,
                    label=f"{r['method']} (val)")
            for _, row in r["val"].iterrows():
                rows.append({"drug": r["drug"], "run": r["run"], "method": r["method"],
                             "round": row["round"], "train_loss": np.nan,
                             "val_loss": row["val_loss"]})
            any_curve = True
    title_drug = recs[0]["drug"]
    ax.set_title(f"{title_drug} — {Path(d).parent.name} — train vs val loss")
    ax.set_xlabel("Round"); ax.set_ylabel("Loss")
    if any_curve:
        ax.legend(fontsize=7); ax.grid(True, ls="--", alpha=0.4)
    else:
        ax.text(0.5, 0.5, "no loss data", ha="center", va="center", transform=ax.transAxes)
    plt.tight_layout()
    safe = f"{title_drug}_{Path(d).parent.name}".replace("/", "_").replace(" ", "_")
    plt.savefig(OUT_DIR / f"loss_curves_{safe}.pdf", bbox_inches="tight")
    plt.show()

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "loss_curves_summary.csv", index=False)
print(f"saved {len(df)} rows to {OUT_DIR / 'loss_curves_summary.csv'}")

## Coverage today

Only runs whose notebooks saved ``*_train_loss_per_round.csv`` appear with a
train curve (currently the Ceftriaxone ``260922`` run). Old runs show a
validation-only curve when their ``*_per_round.csv`` has ``val_loss``.

To fill in other drugs, rerun ``08-Federated-mlp-lr-rf/Drugs/{drug}/federated.ipynb``
on Colab — the main notebook now saves the train-loss CSV and a combined
``loss_curves.pdf`` automatically.